# 04. Feature Engineering & Selection for AML Detection

**Author:** UPADRASTHA VIDHATH  
**Domain:** Anti-Money Laundering (AML) & Financial Intelligence  
**System:** Intelligent Transaction Monitoring & Behavioral Analytics  

---

### Executive Summary
Engineering domain-specific financial crime indicators: velocity scores, account drain ratio, rapid movement flags, and feature selection.


## 1. Imports & Ingestion


In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif

df = pd.read_csv('../dataset/AML_Complete_Transaction_Dataset.csv')


## 2. Financial Crime Domain Feature Creation


In [1]:
# Feature 1: Account Drain Ratio (1.0 indicates complete liquidity liquidation)
df['drain_ratio'] = np.where(df['oldbalanceOrg'] > 0, df['amount'] / df['oldbalanceOrg'], 0)

# Feature 2: Velocity-to-Amount Interaction
df['velocity_amount_risk'] = df['transactionVelocity'] * np.log1p(df['amount'])

# Feature 3: Destination High-Risk Jurisdiction Flag
df['is_high_risk_country'] = (df['countryRiskIndex'] == 3).astype(int)

print("Created 3 domain features successfully.")


## 3. Statistical Feature Importance with ANOVA F-test


In [1]:
feature_cols = ['amount', 'customerRiskScore', 'countryRiskIndex', 'transactionVelocity', 'isStructuring', 'drain_ratio', 'velocity_amount_risk']
X = df[feature_cols].fillna(0)
y = df['isLaundering']

selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X, y)

feature_scores = pd.DataFrame({'Feature': feature_cols, 'Score': selector.scores_}).sort_values('Score', ascending=False)
print("Top Predictive AML Indicators:
", feature_scores)
